# 자연어처리 Homework-1: 단어 벡터 (140점)
### <font color='blue'> 제출 마감일 2026년 4월 7일(화) 23시 59분 59초</font>
#### <font color='red'>답안 작성이나 코딩 위치를 표시하기 위해 작성된 어떤 글도 삭제나 수정하지 말것("채점에 심각한 어려움 초래")</font>

In [ ]:
# ----------------------------
# 파이썬 버전 확인: 3.12 이상
# ----------------------------

import sys
assert sys.version_info[0] == 3
assert sys.version_info[1] >= 12

from platform import python_version
assert int(python_version().split(".")[1]) >= 5, "이 노트북과 동일한 디렉터리에 있는 README.md 파일에 따라 파이썬 버전을 설치 또는 업그레이드 하시오.\
현재 사용중인 파이썬의 버전은"+ python_version()
python_version()

In [ ]:
# ------------------------------------
# 필요한 모든 import 및 설정
# 여기에 다른 코드를 추가하지 마시오.
# ------------------------------------

from gensim.models import KeyedVectors
from gensim.test.utils import datapath
import pprint
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10, 5]

from datasets import load_dataset
imdb_dataset = load_dataset("imdb", name="plain_text")

import re
import numpy as np
import random
import scipy as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import PCA

START_TOKEN = '<s>'
END_TOKEN = '</s>'
NUM_SAMPLES = 150

np.random.seed(0)
random.seed(0)
# ----------------

## 단어 벡터
단어 벡터는 질의응답, 텍스트 생성, 번역 등과 같은 NLP 작업의 핵심 구성 요소로 자주 사용되므로, 단어 벡터의 장단점에 대한 기본적인 이해를 갖는 것이 중요하다. 여기서는 두 가지 유형의 단어 벡터, 즉 **동시발생행렬**(co-occurrence matrix)에서 파생된 단어 벡터와 **GloVe**를 통해 파생된 단어 벡터를 살펴본다.

## 1. 빈도수(Count)-기반 단어벡터 (50점)
**분포 가설(Distributional Hypothesis)**
<br><br>
"단어는 그것이 함께 쓰이는 단어들을 보면 알 수 있다." (Firth, J. R. 1957:11)
<br><br>

### Word-Context Co-Occurence Matrix
동시출현 행렬 $M$: 특정 환경에서 단어들이 얼마나 자주 함께 출현하는지를 계산한다(대칭 행렬). 
- $M_{ij}$ = 모든 문서에서 $w_i$의 context window 내에  $w_j$가 나타나는 횟수
- $w_i$의 문맥창(context window): $w_{i-n} \dots w_{i-1}$과 $w_{i+1} \dots w_{i+n}$.
- $n$: 문맥창(context window)의 크기

**(예시) 고정된 window size n=1을 사용한 동시출현 행렬:**

문서 1: "all that glitters is not gold"  
문서 2: "all is well that ends well"


|     *    | `<s>` | all | that | glitters | is   | not  | gold  | well | ends | `</s>` |
|----------|-------|-----|------|----------|------|------|-------|------|------|-----|
| `<s>`    | 0     | 2   | 0    | 0        | 0    | 0    | 0     | 0    | 0    | 0   |
| all      | 2     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 0    | 0   |
| that     | 0     | 1   | 0    | 1        | 0    | 0    | 0     | 1    | 1    | 0   |
| glitters | 0     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 0    | 0   |
| is       | 0     | 1   | 0    | 1        | 0    | 1    | 0     | 1    | 0    | 0   |
| not      | 0     | 0   | 0    | 0        | 1    | 0    | 1     | 0    | 0    | 0   |
| gold     | 0     | 0   | 0    | 0        | 0    | 1    | 0     | 0    | 0    | 1   |
| well     | 0     | 0   | 1    | 0        | 1    | 0    | 0     | 0    | 1    | 1   |
| ends     | 0     | 0   | 1    | 0        | 0    | 0    | 0     | 1    | 0    | 0   |
| `</s>`      | 0     | 0   | 0    | 0        | 0    | 0    | 1     | 1    | 0    | 0   |


- NLP에서는 입력 시퀀스(예: 문장, 단락 또는 문서)의 시작과 끝을 표시하기 위한 special token으로 일반적으로 `<s>`와 `</s>`을 사용한다. 이러한 토큰은 단어-문맥 동시출현 횟수 계산에 포함된다(예: "`<s>` All that glitters is not gold `</s>`").
- 동시출현 행렬의 행(또는 열)은 단어 간 동시출현을 기반으로 하는 단어 벡터를 제공하지만, 그 크기가 매우 클 수 있다는 것이 문제다. 차원을 축소하기 위해 주성분 분석(PCA)과 유사한 특이값 분해(SVD)를 사용하여 상위 $d$개의 주성분을 선택한다(여기서, $d$가 단어 벡터의 크기가 됨). 
- 이러한 차원 축소는 의미적 관계를 유지한다. 예를 들어, *computer*와 *digital*은 *computer*와 *cherry*보다 더 밀접한 관계를 갖는다.
- 고유값과 특이값 분해(SVD)에 익숙하지 않은 사람은 이클래스 자료실의 "**SP02 특이값분해.pdf**"를 참조하기 바란다.

### 동시출현 단어 임베딩 그래프
#### Large Movie Review Dataset을 사용한 감정 분석(Sentiment Analysis)
**말뭉치 구성**   
- 훈련   25,000 리뷰 및 레이블
- 테스트 25,000 리뷰 및 레이블
- 다른 작업을 위한 레이블 없는 추가 데이터 포함


**`read_corpus()` 함수**
- 영화 리뷰 텍스트 읽어오는 함수
- Special token `<s>` 및 `</s>`를 자동으로 추가하므로 코퍼스에 대한 전처리가 필요 없음

In [ ]:
def read_corpus():
    """ Large Movie Review Dataset에서 파일 읽어오기
        Params:
            category (string): category name
        Return:
            list of lists, 처리된 각 파일의 단어들
    """
    files = imdb_dataset["train"]["text"][:NUM_SAMPLES]
    return [[START_TOKEN] + [re.sub(r'[^\w]', '', w.lower()) for w in f.split(" ")] + [END_TOKEN] for f in files]

읽어온 데이터가 어떻게 생겼는지 한번 보자.

In [ ]:
imdb_corpus = read_corpus()
print(imdb_corpus[:3])
print("corpus size: ", len(imdb_corpus))

### **[문제 1.1]**: `distinct_words` 함수 구현 [코딩] (10점)
- 주어진 코퍼스에 나타나는 서로 다른 단어 모음을 만들고 그 수를 계산하는 메서드를 작성하라.
- [Hint] 입력 `코퍼스`(문자열 리스트의 리스트)를 처리하기 위해 `for` 루프를 사용할 수도 있지만, 일반적으로 더 빠른 파이썬 `리스트 컴프리헨션(List Comprehension)`을 사용하는 것이 좋다. 특히, 리스트의 리스트를 평탄화할 때 유용할 수 있다. 파이썬의 `리스트 컴프리헨션`에 익숙하지 않다면, 다음 [링크](https://coderwall.com/p/rcmaea/flatten-a-list-of-lists-in-one-line-in-python)를 참고하라.
- 반환되는 `corpus_words`는 정렬되어 있어야 한다. 파이썬의 `sorted` 함수를 사용하여 정렬할 수 있다.
- 중복되는 단어를 제거하기 위해 파이썬 `set`을 사용하는 것도 도움이 될 수 있다.

In [ ]:
def distinct_words(corpus):
    """ 코퍼스의 서로 다른 단어(distinct words)를 찾는 함수.
        Params:
            corpus (list of list of strings): 코퍼스 
        Return:
            corpus_words (list of strings): 코퍼스 내 다른 단어들의 정렬된 리스트 
            n_corpus_words (int): 코퍼스 내 다른 단어 수
    """
    corpus_words = []
    n_corpus_words = -1
    
    # ------------------
    # 여기에 코딩하시오.

    
    
    # ------------------

    return corpus_words, n_corpus_words

In [ ]:
# --------------------------------------------------------------
# 이 cell을 실행하여 위에서 작성한 코드의 정확성을 테스트하시오.
# [주의] 이 코드는 정확성에 대한 완벽한 검사를 보장하지 않는다.
# --------------------------------------------------------------

# 테스트에 사용할 toy corpus 정의:
test_corpus = ["{} All that glitters isn't gold {}".format(START_TOKEN, END_TOKEN).split(" "), "{} All's well that ends well {}".format(START_TOKEN, END_TOKEN).split(" ")]
test_corpus_words, num_corpus_words = distinct_words(test_corpus)

# 정답
ans_test_corpus_words = sorted([START_TOKEN, "All", "ends", "that", "gold", "All's", "glitters", "isn't", "well", END_TOKEN])
ans_num_corpus_words = len(ans_test_corpus_words)

# 단어수 테스트
assert(num_corpus_words == ans_num_corpus_words), f"n_corpus_words 계산 틀림. 정답: {ans_num_corpus_words}. 제출: {num_corpus_words}"

# 단어의 정확성 테스트
assert(test_corpus_words == ans_test_corpus_words), f"corpus_words 계산 틀림.\n정답: {str(ans_test_corpus_words)}\n제출: {str(test_corpus_words)}"

# 테스트 통과 출력
print ("-" * 80)
print("모든 테스트 통과!")
print ("-" * 80)

### **[문제 1.2]**: `compute_co_occurrence_matrix` [코딩] (10점)
- 동시출현 행렬을 만드는 코드를 작성한다.
- `window_size`: 윈도우 크기 변수(기본값 4)
- `numpy` 라이브러리만 사용하여 벡터, 행렬 및 텐서를 구현한다.
- `numpy` 사용이 익숙하지 않으면, 깃허브의 저장소 nlp2026의 실습 코드 중 `python_tutorial` 디렉토리 내의 `00-2.NumPy.ipynb`를 참고한다.

**[주의]** 
- 문서 내 각 단어는 window 내에서 중앙에 위치해야 한다.
- 가장자리에 있는 단어들은 더 적은 수의 동시출현 단어를 가지게 된다.
- (예) window_size=4 이고 입력 문서가 "\<s> All that glitters is not gold \</s>"이면, "All"은 "\<s>", "that", "glitters", "is", "not"과 동시출현한다.

In [ ]:
def compute_co_occurrence_matrix(corpus, window_size=4):
    """ 주어진 코퍼스와 window_size(기본값 4)에 대하여 동시출현 행렬을 계산한다.
    
        Params:
            corpus (list of list of strings): 코퍼스
            window_size (int): context window 크기
        Return:
            M (대칭인 numpy 행렬. 크기 = (코퍼스 내 고유 단어 수 x 코퍼스 내 고유 단어 수)): 단어 수 카운트 기반의 동시출현 행렬 
                행과 열의 단어 순서는 distinct_words 함수에서 반환된 단어들의 순서와 동일해야 한다.
            word2ind (dict): 단어를 행렬 M에 대한 인덱스로 매핑하는 딕셔너리 (예: row 또는 column number).
    """
    words, n_words = distinct_words(corpus)
    M = np.zeros((n_words, n_words))
    word2ind = {word: idx for idx, word in enumerate(words)}
    
    # ------------------
    # 여기에 코딩하시오.


    
    # ------------------

    return M, word2ind

In [ ]:
# --------------------------------------------------------------
# 이 cell을 실행하여 위에서 작성한 코드의 정확성을 테스트하시오.
# [주의] 이 코드는 정확성에 대한 완벽한 검사를 보장하지 않는다.
# --------------------------------------------------------------

# Toy corpus를 정의하고 과제 작성자의 co-occurrence matrix를 받음
test_corpus = [f"{START_TOKEN} All that glitters isn't gold {END_TOKEN}".split(" "), f"{START_TOKEN} All's well that ends well {END_TOKEN}".split(" ")]
M_test, word2ind_test = compute_co_occurrence_matrix(test_corpus, window_size=1)

# Correct M and word2ind
M_test_ans = np.array( 
    [[0., 0., 0., 0., 0., 0., 1., 0., 0., 1.,],
     [0., 0., 1., 1., 0., 0., 0., 0., 0., 0.,],
     [0., 1., 0., 0., 0., 0., 0., 0., 1., 0.,],
     [0., 1., 0., 0., 0., 0., 0., 0., 0., 1.,],
     [0., 0., 0., 0., 0., 0., 0., 0., 1., 1.,],
     [0., 0., 0., 0., 0., 0., 0., 1., 1., 0.,],
     [1., 0., 0., 0., 0., 0., 0., 1., 0., 0.,],
     [0., 0., 0., 0., 0., 1., 1., 0., 0., 0.,],
     [0., 0., 1., 0., 1., 1., 0., 0., 0., 1.,],
     [1., 0., 0., 1., 1., 0., 0., 0., 1., 0.,]]
)
ans_test_corpus_words = sorted([START_TOKEN, "All", "ends", "that", "gold", "All's", "glitters", "isn't", "well", END_TOKEN])
word2ind_ans = dict(zip(ans_test_corpus_words, range(len(ans_test_corpus_words))))

# word2ind 테스트
assert (word2ind_ans == word2ind_test), f"생성한 word2ind가 틀림:\n정답: {word2ind_ans}\n오답: {word2ind_test}"

# 행렬 M 크기 테스트
assert (M_test.shape == M_test_ans.shape), f"행렬 M이 크기가 틀렸음.\n정답: {M_test.shape}\n오답: {M_test_ans.shape}"

# M 값의 정확성 테스트
for w1 in word2ind_ans.keys():
    idx1 = word2ind_ans[w1]
    for w2 in word2ind_ans.keys():
        idx2 = word2ind_ans[w2]
        student = M_test[idx1, idx2]
        correct = M_test_ans[idx1, idx2]
        if student != correct:
            print("정답 M:")
            print(M_test_ans)
            print("오답 M: ")
            print(M_test)
            raise AssertionError(f"행렬 M에서 index ({idx1}, {idx2})=({w1}, {w2})의 count 오류. 계산된 오답은 {student}, 그러나 정답은 {correct}.")

# 테스트 통과 출력
print ("-" * 80)
print("모든 테스트 통과!")
print ("-" * 80)

### **[문제 1.3]**: `reduce_to_k_dim` 구현 [코딩] (10점)

- 동시출현 행렬에서 k 차원 임베딩을 생성하기 위한 차원 축소를 수행한다. 
- SVD를 사용하여 상위 k 개의 성분을 취하고 k 차원 임베딩으로 이루어진 새로운 행렬을 만든다. 
  - SVD는 M을 다음과 같이 분해한다:
  $$M = U \Sigma V^T$$
- **[주목]** numpy, scipy와 scikit-learn은 모두 나름대로의 SVD 구현을 제공한다.
  - 그러나, scipy와 scikit-learn만 **Truncated SVD**(절단 SVD)를 제공하며, sklearn만 대규모 Truncated SVD를 위한 효율적 무작위화 알고리즘을 제공한다.
  - 따라서, [sklearn.decomposition.TruncatedSVD](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)를 사용할 것을 권장한다.
- Truncated SVD는 $U \Sigma$ 를 반환하며(code cell에서는 `U*S`로 표기됨), 이것이 차원축소된 임베딩 행렬이다.

In [ ]:
def reduce_to_k_dim(M, k=2):
    """ 크기가 (num_corpus_words, num_corpus_words)인 동시출현 행렬을 (num_corpus_words, k)인 행렬로 차원 축소한다.
        Scikit-Learn의 SVD 함수를 사용한다:
            - http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html
    
        Params:
            M (차원이 (number of unique words in the corpus , number of unique words in the corpus)인 numpy 행렬): 단어 빈도수 기반 동시출현 행렬
            k (int): 차원 축소된 단어 임베딩의 크기
        Return:
            M_reduced (차원이 (number of corpus words, k)인 numpy 행렬): k-차원 단어 임베딩의 행렬(U*S)
    """    
    n_iters = 10    # `TruncatedSVD` 호출시 이 매개변수를 사용하시오.
    M_reduced = None
    print(f"{M.shape[0]} 개 단어들에 대한 Truncated SVD 수행중...")
    
    # ------------------
    # 여기에 코딩하시오.


    
    # ------------------

    print("완료.")
    return M_reduced

In [ ]:
# --------------------------------------------------------------
# 이 cell을 실행하여 위에서 작성한 코드의 정확성을 테스트하시오.
# [주의] 이 코드는 정확성에 대한 완벽한 검사를 보장하지 않는다.
# --------------------------------------------------------------


# Define toy corpus and run student code
test_corpus = [f"{START_TOKEN} All that glitters isn't gold {END_TOKEN}".split(" "), f"{START_TOKEN} All's well that ends well {END_TOKEN}".split(" ")]
M_test, word2ind_test = compute_co_occurrence_matrix(test_corpus, window_size=1)
M_test_reduced = reduce_to_k_dim(M_test, k=2)

# Test proper dimensions
assert (M_test_reduced.shape[0] == 10), f"M_reduced의 행 수 = {M_test_reduced.shape[0]} 개; 정답은 10"
assert (M_test_reduced.shape[1] == 2), f"M_reduced의 열 수 = {M_test_reduced.shape[1]} 개; 정답은 2"

# 테스트 통과 출력
print ("-" * 80)
print("모든 테스트 통과!")
print ("-" * 80)

### **[문제 1.4]**: `plot_embeddings` 구현 [코딩] (10 points)
- 2D 벡터 집합을 2D 공간에 나타내는 함수를 구현한다.
- 그래프를 그리기 위해 Matplotlib (`plt`)을 사용한다.
- [참고자료]
  - 실습자료: nlp2026/python_tutorial/00-4.Matplotlib.ipynb
  - [matplotlib scatter plot annotate / set text at / label each point](https://web.archive.org/web/20190924160434/https://www.pythonmembers.club/2018/05/08/matplotlib-scatter-plot-annotate-set-text-at-label-each-point/)
  - [the Matplotlib gallery](https://matplotlib.org/gallery/index.html)

In [ ]:
def plot_embeddings(M_reduced, word2ind, words):
    """ "words" list에 명시된 단어들의 임베딩을 산점도(Scatter Plot)로 그리는 함수를 구현한다.
        [주의] M_reduced 또는 word2ind에 열거된 모든 단어를 그리지 말 것.
        각 단어에 라벨을 추가한다.
        
        Params:
            M_reduced (크기 (number of unique words in the corpus x 2)인 numpy 행렬): 2-차원 단어 임베딩의 행렬
            word2ind (dict): 단어를 행렬 M에 대한 인덱스로 매핑하는 딕셔너리
            words (list of strings): 우리가 시각화할 단어들의 list
    """

    # ------------------
    # 여기에 코딩하시오.



    # ------------------

In [ ]:
# --------------------------------------------------------------
# 이 cell을 실행하여 위에서 작성한 코드의 정확성을 테스트하시오.
# [주의] 이 코드는 정확성에 대한 완벽한 검사를 보장하지 않는다.
# --------------------------------------------------------------

print ("-" * 80)
print ("출력된 그림:")

M_reduced_plot_test = np.array([[1, 1], [-1, -1], [1, -1], [-1, 1], [0, 0]])
word2ind_plot_test = {'test1': 0, 'test2': 1, 'test3': 2, 'test4': 3, 'test5': 4}
words = ['test1', 'test2', 'test3', 'test4', 'test5']
plot_embeddings(M_reduced_plot_test, word2ind_plot_test, words)

print ("-" * 80)

### **[문제 1.5]**:  동시출현 그래프에 대한 분석 [서술형 답] (10점)

지금까지의 과정을 종합해보자:
1. Large Movie Review 코퍼스를 사용하여 동시출현 행렬을 계산했다(window_size=4).
2. TruncatedSVD를 사용하여 각 단어의 2차원 임베딩을 계산했다.  

이제, 생성된 단어 임베딩을 그림으로 나타내려고 한다.
TruncatedSVD는 $U \Sigma$ (또는 `U*S`)를 반환하므로, 생성된 단어 임베딩의 그림을 그리기 전에 모든 벡터가 단위 원 주위에 나타나도록(즉, 방향성 근접성을 갖도록) 반환된 벡터를 정규화해야 한다.

In [ ]:
# --------------------------------------------
# 주어진 단어에 대한 단어 임베딩 산점도 그리기
# --------------------------------------------
imdb_corpus = read_corpus()
M_co_occurrence, word2ind_co_occurrence = compute_co_occurrence_matrix(imdb_corpus)
M_reduced_co_occurrence = reduce_to_k_dim(M_co_occurrence, k=2)

# 각 행이 단위 길이를 갖도록 정규화(normalize)
M_lengths = np.linalg.norm(M_reduced_co_occurrence, axis=1)
M_normalized = M_reduced_co_occurrence / M_lengths[:, np.newaxis] # broadcasting

words = ['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']

plot_embeddings(M_normalized, word2ind_co_occurrence, words)

**출력된 그림이 images 폴더의 "question_1.5.png"와 일치하는지 확인하자. 만일 일치하지 않는다면, 폴더에 있는 "question_1.5.png" 이미지를 보면서 다음 질문에 답하시오.**

a. 2차원 임베딩 공간에서 함께 묶이는(Clustering되는) 단어 그룹을 최소 두 개 이상 찾고 각 그룹(Cluster)에 대하여 간단히 설명하라.

#### <font color="red">여기에 답을 작성하시오.</font>

b. 함께 묶여야 한다고 생각하지만 그렇지 않은 것은 무엇인가? 최소 두 가지 예를 들어 설명하시오.

#### <font color="red">여기에 답을 작성하시오.</font>

## Part 2: Prediction-Based Word Vectors (90점)

- 강의 시간에 논의했듯이, word2vec과 GloVe 같은 예측 기반 단어 벡터가 더 나은 성능을 보여준다. 
- 다음 Cell을 실행하여 GloVe 벡터를 메모리에 로드하자
- [참고] 이 Cell을 처음 실행하는 경우(즉, 임베딩 모델을 처음 다운로드할 때), 실행 시간은 몇 분 정도 소요된다. 이전에 실행한 적이 있는 경우, 다시 실행하면 모델을 다시 다운로드하지 않고 로드되며, 약 1~2분 정도 소요된다.

In [ ]:
def load_embedding_model():
    """ GloVe 벡터 로드하기
        로드할 임베딩: 2B 개의 트윗, 27B 개의 토큰, 1.2M 개의 어휘를 기반으로 사전 학습된 GloVe 벡터(대소문자 구분 없음).
        Return:
            wv_from_bin: 임베딩 400000 개, 각 임베딩의 길이는 200
    """
    import gensim.downloader as api
    
    wv_from_bin = api.load("glove-wiki-gigaword-200")
    print(f"{len(list(wv_from_bin.index_to_key))}개의 어휘 임베딩을 로드했음.")
    return wv_from_bin
wv_from_bin = load_embedding_model()

### 단어 임베딩의 차원 축소
GloVe 임베딩을 동시출현 행렬의 임베딩과 직접 비교해보자. 메모리 부족을 피하기 위해, 40000개의 샘플로 작업할 것이다.  
다음 cell에서는 아래와 같은 작업을 수행한다:
1. 40000개의 GloVe 벡터 샘플을 행렬 M에 넣는다.
2. 벡터의 차원을 200에서 2로 줄이기 위해 Truncated SVD 함수를 사용하는 `reduce_to_k_dim`를 호출한다.

In [ ]:
# [주의] 다음 함수 내에서 사용되는 변수 'words'는 [문제 1.5]에서 정의된 'words'와 다른 지역 변수이다. 
# 앞에서 정의된 'words'는 그래프로 비교할 단어 목록을 담고 있다.

def get_matrix_of_vectors(wv_from_bin, required_words):
    """ GloVe 벡터를 행렬 M에 집어 넣기.
        Param:
            wv_from_bin: KeyedVectors 객체(파일에서 로드된 400000개의 GloVe 벡터)
            required_words: 그래프로 비교할 단어 목록
        Return:
            M: 벡터를 포함하고 있는 크기 (num words, 200)인 numpy 행렬
            word2ind: 각 단어를 M의 행(row)으로 매핑하는 딕셔너리
    """
    import random
    
    words = list(wv_from_bin.index_to_key)
    print("단어 Shuffling중 ...")
    random.seed(225)
    random.shuffle(words)
    print(f"{len(words)} 단어를 word2ind와 행렬 M에 넣기...")
    word2ind = {}
    M = []
    curInd = 0
    for w in words:
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
    for w in required_words:
        if w in words:
            continue
        try:
            M.append(wv_from_bin.get_vector(w))
            word2ind[w] = curInd
            curInd += 1
        except KeyError:
            continue
    M = np.stack(M)
    print("완료.")
    return M, word2ind

In [ ]:
# -----------------------------------------------------------------
# 이 Cell을 사용하여 200차원 단어 임베딩을 k 차원으로 축소
# -----------------------------------------------------------------
M, word2ind = get_matrix_of_vectors(wv_from_bin, words)
M_reduced = reduce_to_k_dim(M, k=2)

# 각 행 벡터의 길이가 1이 되도록 정규화(Normalization) 
M_lengths = np.linalg.norm(M_reduced, axis=1)
M_reduced_normalized = M_reduced / M_lengths[:, np.newaxis] # broadcasting

In [ ]:
M_reduced

**[주목]** 로컬 컴퓨터에서 메모리 부족 오류가 발생하는 경우, 다른 애플리케이션을 종료하여 장치의 메모리를 확보해 보라. 메모리 확보를 위해 컴퓨터를 재시작하는 것도 도움이 될 수 있다. 그런 다음 즉시 Jupyter Notebook을 실행하고 단어 벡터가 제대로 로드되는지 확인하라. 이 과정을 거친 후에도 임베딩을 로컬 컴퓨터에 로드하는 데 문제가 계속 발생하는 경우, 교수에게 문의하시오.

### **[문제 2.1]**:  GloVe Plot에 대한 분석 [서술형 답] (10점)

아래 cell을 실행하여 `['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']` 에 대한 2D GloVe 임베딩을 산점도 그래프로 나타내시오(images\question_1.5.png와 같은 형식).

In [ ]:
words = ['movie', 'book', 'mysterious', 'story', 'fascinating', 'good', 'interesting', 'large', 'massive', 'huge']

plot_embeddings(M_reduced_normalized, word2ind, words)

**출력된 그림이 "images\question_2.1.png"와 같은지 비교하시오. 만일 다르면, "images\question_2.1.png"를 사용하여 다음 두 질문에 답하시오.**

a. 이 그래프는 앞에서 동시 발생 행렬을 기반으로 생성된 그래프와 어떤 점에서 다른가? 어떤 점에서 비슷한가?

#### <font color="red">여기에 답을 작성하시오.</font>

b. GloVe 그림(question_2.1.png)이 이전의 동시출현 행렬에서 생성된 그림(question_1.5.png)과 다른 이유는 무엇일까?

#### <font color="red">여기에 답을 작성하시오.</font>

### 코사인(Cosine) 유사도
이제 단어 벡터를 얻게 되었고, 이 벡터들을 사용하여 단어들 간의 코사인 유사도를 계산할 수 있게 되었다.
코사인 유사도에서는 두 벡터의 사잇각 대신 코사인 함수 값 $(-1, 1)$을 유사도로 사용한다.

### **[문제 2.2]**:  다의어(Polyseme) [코딩] (10점)
- 다의어(Polyseme)와 동음이의어(homonym)는 두 개 이상의 의미를 가진 단어를 말한다. 다의어와 동음이의어의 차이는 [wiki page](https://en.wikipedia.org/wiki/Polysemy)를 참조하시오.
- **[질문]** 아래 cell에 제시된 다의어 candidates 목록에 대하여, 코사인 유사도에 따라 각각 가장 유사한 상위 10개 단어를 찾으시오.
- **[참고]** 가장 유사한 상위 10개 단어를 얻으려면 `wv_from_bin.most_similar(word)` 함수를 사용해야 할 것이다. 이 함수는 주어진 단어와의 코사인 유사도를 기준으로 어휘에 있는 다른 모든 단어의 순위를 매긴다. 더 자세한 내용은 [GenSim documentation](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.FastTextKeyedVectors.most_similar)를 참조하라.


In [ ]:
candidates=['leaves','scoop','bank','crane','bass','bat','apple','fly','spring']

# ------------------
# 여기에 코딩하시오.




# ------------------

### **[문제 2.3]**: 동의어(Synonyms) 및 반의어(Antonyms) [코딩 + 서술형 답] (10점)
- 코사인 유사도를 사용할 수 있을 때, 종종 코사인 거리(1 - 코사인_유사도)로 생각하는 것이 더 편리한 경우가 있다.
- 다음 조건을 만족하는 세 단어로 된 묶음 $(w_1,w_2,w_3)$들을 제시된 예 이외에 추가로 3 묶음 이상 더 찾으시오:
  - $w_1$과 $w_2$는 동의어이고 $w_1$과 $w_3$는 반의어
  - Cosine_Distance($w_1$, $w_2$) > Cosine_Distance($w_1$, $w_3$)
  - (예) (happy, cheerful, sad)
- **[질문]** 위의 현상은 우리의 직관적 이해에 반하는 결과이다. 왜 이러한 반직관적(anti-intuitive) 결과가 나올 수 있는지 설명하시오.
- **[참고]** 두 단어 사이의 코사인 거리 측정을 위해, `wv_from_bin.distance(w1, w2)` 함수를 사용해야 할 수 있다. 자세한 내용은 [GenSim documentation](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.FastTextKeyedVectors.distance)를 참고하시오.

In [ ]:
# ------------------
# 여기에 코딩하시오.



# ------------------

#### <font color="red">여기에 답을 작성하시오.</font>

### **[문제 2.4]**: 단어 벡터를 사용한 유추 실험(Analogy Test) [서술형 답] (10점)
- 단어 벡터는 유추 문제를 해결하는 데 유용하다.
- (예) "남자 : 할아버지 :: 여자 : x" (즉, 남자와 할아버지 관계는 여자와 x 관계와 같다)라는 유추에서 x는 무엇일까?
- 아래 예시에서는 **GenSim documentation**의 `most_similar` 함수를 사용하여 단어 벡터로 x를 찾는 방법을 보여준다.
  - `positive`는 유추 식의 `+`할 단어 목록이고, `negative`는 `-`할 단어이다. 
  - 이 함수는 positive 목록의 단어들과 가장 유사하고 negative 목록의 단어와 가장 유사하지 않은 단어를 찾는다(입력 단어는 제외하는데, 입력 단어가 가장 유사한 경우가 많기 때문이다. 자세한 내용은 이 [논문](https://www.aclweb.org/anthology/N18-2039.pdf)을 참조하시오).
- 유추의 답은 코사인 유사도(반환되는 수치 값이 가장 큰 값)가 가장 높은 단어이다.

In [ ]:
# 이 Cell을 실행하여 유추 문제 man : grandfather :: woman : x 의 답을 구한다.
pprint.pprint(wv_from_bin.most_similar(positive=['woman', 'grandfather'], negative=['man']))

**[질문]** 단어 벡터 $m$, $g$, $w$, 그리고 $x$는 각각 `man`, `grandfather`, `woman`, 그리고 답을 나타낸다. 세 벡터 $m$, $g$, $w$, 그리고 벡터 연산자 $+$ 및 $-$ 만을 사용하여 답을 작성하여야 한다. $x$와의 코사인 유사도를 최대화시키기 위한 표현(수식)을 LaTex로 작성하시오.

#### <font color="red">여기에 답을 작성하시오.</font>

### **[문제 2.5]**: 유추 단어 찾기 [코딩 + 서술형 답] (15점) 
a. 앞의 예에서, 유추의 정답은 "grandmother"임을 바로 알 수 있다. 그러나, `most_similar` 함수는 "granddaughter", "daughter", 또는 "mother" 같은 답들을 준다. 왜 그럴까? 직관적인 설명을 제시하시오.

#### <font color="red">여기에 답을 작성하시오.</font>

b. 이와 같이 기대하는 단어의 우선 순위가 제일 높게 나오는 벡터 유추의 예를 제시하시오. x:y :: a:b 형태가 되도록, x, y, a, b를 명시하면 된다. 여기서 b가 찾으려하는 단어이다.

[주목] 잘 성립하는 유추를 찾기 위해 여러 유추 사례를 시험해 봐야 할 수도 있다! 

In [ ]:
# 예시: x, y, a, b = ("", "", "", "")
# ------------------
# 여기에 코딩하시오.



# ------------------

# 만든 답의 유추가 잘 되는지 확인하시오:
assert wv_from_bin.most_similar(positive=[a, y], negative=[x])[0][0] == b

In [ ]:
wv_from_bin.most_similar(positive=[a, y], negative=[x])

**[질문]** 위에서 찾은 유추가 어떻게 잘 작동하는지 간략히 설명하시오.

#### <font color="red">여기에 답을 작성하시오.</font>

### **[문제 2.6]**: 틀린 유추 [코딩 + 서술형 답] (15점) 
a. 아래에서, 우리가 의도한 유추 "hand : glove :: foot : **sock**"를 기대했지만 답이 틀리게 나왔다. 이 특정 유추가 이러한 결과로 나타난 이유를 설명해 보라.

In [ ]:
pprint.pprint(wv_from_bin.most_similar(positive=['foot', 'glove'], negative=['hand']))

#### <font color="red">여기에 답을 작성하시오.</font>

b. 이와 같이 벡터에 대한 유추가 성립하지 않는 예를 제시하시오. 원래 의도한 유추는 무엇이었는데 실제 결과는 어떻게 틀리게 나왔는지를 설명하시오. 

In [ ]:
# 예시: x, y, a, b = ("", "", "", "")
# ------------------
# 여기에 코딩하시오.



# ------------------
pprint.pprint(wv_from_bin.most_similar(positive=[a, y], negative=[x]))
assert wv_from_bin.most_similar(positive=[a, y], negative=[x])[0][0] != b

#### <font color="red">여기에 답을 작성하시오.</font>

### **[문제 2.7]**: 단어 벡터의 편향에 대한 분석 [서술형 답] (10점) 
단어 임베딩에 내재된 편견(성별, 인종, 성적 지향 등)을 인지하는 것이 중요하다. 이러한 편견은 해당 모델을 사용하는 애플리케이션을 통해 고정관념을 심화시킬 수 있기 때문에 위험하다. 아래 셀을 실행하여,   
(a) "남성" 및 "직업"과 가장 유사하고 "여성"과 가장 유사하지 않은 단어는 무엇인지,   
(b) "여성" 및 "직업"과 가장 유사하고 "남성"과 가장 유사하지 않은 단어는 무엇인지 살펴보라. 여성과 관련된 단어 목록과 남성과 관련된 단어 목록의 차이점을 지적하고, 이러한 차이점이 어떻게 성별 편견을 반영하는지 설명하시오.

In [ ]:
# 이 cell을 실행하시오.
# 여기서 `positive`는 이것과 유사해야 할 단어 목록을 나타내며, `negative`는 이것과 가장 유사하지 않은 단어 목록을 나타냄 

pprint.pprint(wv_from_bin.most_similar(positive=['man', 'profession'], negative=['woman']))
print()
pprint.pprint(wv_from_bin.most_similar(positive=['woman', 'profession'], negative=['man']))

#### <font color="red">여기에 답을 작성하시오.</font>

### **[문제 2.8]**: 단어 벡터의 편향에 대한 독립적 분석 [코딩 + 서술형 답] (10점) 
`most_similar` 함수를 사용하여 벡터들이 보이는 어떤 편향성을 보여주는 유추 쌍을 찾으시오. 찾은 편향의 예를 간단히 설명하시오. 

In [ ]:
# 예시: x, y, a, b = ("", "", "", "")
# ------------------
# 여기에 코딩하시오.



# ------------------
pprint.pprint(wv_from_bin.most_similar(positive=[a, y], negative=[x]))
# ------------------

#### <font color="red">여기에 답을 작성하시오.</font>

# <font color="blue"> 과제 제출 요령</font>
1. Jupyter Notebook 좌측 상단의 저장 버튼을 클릭한다.
2. Edit -> "Clear Outputs of All Cells"를 선택하여 모든 셀의 출력을 지운다.
3. Run -> "Run All Cells"를 선택하여 모든 셀을 순서대로 실행시킨다(몇 분 정도 소요됨).
4. 모든 셀의 실행이 완료되면 File -> "Save and Export Notebook as"에서 -> PDF를 선택한다("nbconvert 실패: Pandoc을 찾을 수 없습니다"와 같은 오류가 표시되면 먼저 HTML로 저장한다).
5. HTML로 저장한 경우: 저장된 HTML 파일을 웹 브라우저에서 열고 인쇄 옵션에서 PDF로 저장하여 PDF 파일을 만들 수 있다.
6. <font color='blue'>모든 풀이 과정, 특히 코딩 부분이 PDF 파일에 제대로 표시되는지 확인할 것.</font> 코드 셀에서 줄 바꿈이 되지 않아 코드가 잘리는 것은 괜찮다.
7. 채점자는 PDF 파일만 보게 되므로 이 점에 주의할 것!
8. 최종 PDF 파일을 이클래스에 제출한다.<font color='blue'>파일명 작성 요령: "HW1_홍길동_학번.pdf" => 이름에 의한 Sorting에 문제가 없도록 주의한다.</font>
9. <font color='red'>파일명의 한글이 깨지는 것은 감점 요인이므로 주의!(특히 Apple 계열 사용자) </font>